## **Camada Silver - silver.tb_info_filmes (origem: tb_movies_info)**

In [0]:
# Importa as funções do PySpark que serão utilizadas nas transformações
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.getOrCreate()

# Cria o database/schema "silver"
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Lê a tabela de filmes da camada Bronze
df_filmes = spark.table("bronze.tb_movies_info")

# Exibe algumas linhas para conferirmos os dados de origem
display(df_filmes.limit(20))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T07:31:36.959Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T07:31:36.959Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T07:31:36.959Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T07:31:36.959Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T07:31:36.959Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-10-16T07:31:36.959Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-10-16T07:31:36.959Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-10-16T07:31:36.959Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

In [0]:
# Normaliza a coluna 'status' original: remove ruídos, hífens e padroniza a caixa
df_filmes = df_filmes.withColumn("status_normalizado", F.trim(F.lower(F.regexp_replace(F.col("status"), "-", " "))))

# Tradução dos termos e padronização dos não mapeáveis como "Não Informado"
df_filmes = df_filmes.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "released", "Lançado")
     .when(F.col("status_normalizado") == "post production", "Pós-Produção")
     .when(F.col("status_normalizado") == "in production", "Em Produção")
     .when(F.col("status_normalizado") == "planned", "Planejado")
     .when(F.col("status_normalizado") == "rumored", "Rumores")
     .when(F.col("status_normalizado") == "canceled", "Cancelado")
     .otherwise("Não Informado")
)

# Cria uma janela agrupando os registros pelo ID do filme e ordenando pela data de ingestão da mais recente para a mais antiga
janela_filmes = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

# Ordena cada filme pela ingestão mais recent e mantém apenas a versão mais recente de cada filme
df_filmes = (df_filmes.withColumn("rn", F.row_number().over(janela_filmes)).filter(F.col("rn") == 1).drop("rn"))

# Converte a coluna original 'release_date' testando os diferentes padrões
df_filmes = df_filmes.withColumn(
    "release_date_tratada",
    F.coalesce(
        F.try_to_date(F.col("release_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("release_date"), "MM-dd-yyyy"),
        F.try_to_date(F.col("release_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("release_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("release_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("release_date"), "MM/dd/yyyy")
    )
)

# Criar a coluna 'ano_lancamento', extraída da data tratada
df_filmes = df_filmes.withColumn("ano_lancamento", F.year(F.col("release_date_tratada")))

# Mapeamento de Colunas, renomeação para Português e tipagem

df_silver = df_filmes.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("release_date_tratada").alias("data_lancamento"),
    F.col("ano_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),

    # Mantém apenas códigos de idioma com duas letras
    F.when(
        F.col("original_language").rlike(r"^[A-Za-z]{2}$"),
        F.lower(F.col("original_language"))
    ).otherwise(None).alias("idioma_original"),

    F.col("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao")
)

# Conferência visual dos dados finais da camada Silver
# display(df_silver.limit(20))

# Gravação na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

# Confere a tabela após a gravação
display(spark.table("silver.tb_info_filmes").limit(20))

id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null
1000088,Monsieur le Maire,Monsieur le Maire,2023-11-01,2023,0,fr,Pós-Produção,null,null
1000096,Critical Keertanegalu,Critical Keertanegalu,2022-07-15,2022,129,kn,Lançado,"A Lawyer called Kariyappa, he has been instrumental in filing the flaws in society, During IPL season, Gambling became a reality and millions became victims of the year. Realising that he comes to court to file an appeal against IPL for the of ban the IPL. He describes the four gambling events happened in Karnataka. All this as lawyer Kariyappa puts it infront of judge.",null
1000099,Chowdhury Rajbari,Chowdhury Rajbari,2017-12-08,2017,131,bn,Lançado,"Giaa and Arnab are a married couple, who visit Arnab's ancestral palace 'Chowdhury Rajbari' for a romantic getaway. Once there, Giaa experiences strange occurrences and memories of a past life. In her search for truth, Giaa comes across Dr. Oindrila, who assists her in her search. But trouble starts when they come across more questions, instead of answers.",null
1000116,Don't Wait for Me,אל תחכי לי,2021-10-21,2021,88,he,Lançado,Matan arrives in a crime neighborhood after his father goes bankrupt and loses their home.,null
1000130,Hounds,Les meutes,2023-07-19,2023,94,ar,Lançado,"In the working class suburbs of Casablanca. Hassan and Issam, father and son, try to survive from day to day, doing small deals for the local underworld. One night, a man they were supposed to kidnap dies accidentally in their car. Hassan and Issam find themselves with a corpse to dispose of. Then begins a long night through the underworld of the city.",null
1000173,Nightman,Nightman,2023-09-07,2023,101,en,Lançado,A young pregnant woman discovers that her perfect boyfriend may be a dangerous sleepwalker when she finds out of strange disappearances and murders taking place in the region.,null
1000221,#LookAtMe,#LookAtMe,2022-07-23,2022,108,en,Lançado,"A Youtuber posts an irreverent video trolling a megachurch pastor, in defence of his gay twin brother. He is vilified by society, tried in court, and pitted against a culture that threatens to destroy his family.",null


## **Camada Silver - silver.tb_financeiro_filmes (origem: tb_movies_financials)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_financeiro_filmes")

# Lê os dados financeiros da camada Bronze
df_financeiro = spark.table("bronze.tb_movies_financials")

# Exibe o schema para identificarmos os tipos das colunas
# df_financeiro.printSchema()

# Exibe algumas linhas para verificarmos o formato dos valores monetários
display(df_financeiro.limit(20))

id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-10-16T13:11:29.210Z
299536,300000000,2052415039,2026-10-16T13:11:29.210Z
299534,356000000,2800000000,2026-10-16T13:11:29.210Z
475557,55000000,1074458282,2026-10-16T13:11:29.210Z
271110,250000000,Não Informado,2026-10-16T13:11:29.210Z
284054,200000000,1349926083,2026-10-16T13:11:29.210Z
284052,180000000,676343174,2026-10-16T13:11:29.210Z
315635,175000000,880166924,2026-10-16T13:11:29.210Z
283995,200000000,863756051,2026-10-16T13:11:29.210Z
297761,175000000,746846894,2026-10-16T13:11:29.210Z


In [0]:
# Lê a cotação do dólar da camada Bronze
# df_cotacao = spark.table("bronze.tb_cotacao_dolar")

# Exibe o schema e algumas linhas para conferirmos os dados
# df_cotacao.printSchema()
# display(df_cotacao.limit(20))

# Carrega as tabelas Bronze
df_cotacao = spark.table("bronze.tb_cotacao_dolar")
df_financeiro = spark.table("bronze.tb_movies_financials")

# Obtém a taxa mais recente do dólar
taxa_dolar = float(df_cotacao
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)

In [0]:
# Renomeando os nomes das colunas
df_tratado = df_financeiro.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.col("budget").alias("orcamento_raw"),
    F.col("revenue").alias("receita_raw")
)
# Tratar valores textuais de ausência como NULL antes da conversão
textos_ausentes = ["unknown", "não informado", "na", "n/a", "none", "null", ""]
for col_raw in ["orcamento_raw", "receita_raw"]:
    df_tratado = df_tratado.withColumn(
        col_raw,
        F.when(F.trim(F.lower(F.col(col_raw))).isin(textos_ausentes), None)
         .otherwise(F.col(col_raw))
    )
# Higienizar para remover símbolos de moedas (USD, R$, $), pontuações de milhar e textos
for col_raw in ["orcamento_raw", "receita_raw"]:
    # Mantém apenas dígitos numéricos e o ponto decimal
    df_tratado = df_tratado.withColumn(col_raw,
        F.regexp_replace(F.col(col_raw), r"[^0-9.]", "")
    )
# Converter para tipo numérico decimal apropriado e tratar <= 0 como ausente (NULL)
df_tratado = (df_tratado
    .withColumn("orcamento_usd", F.expr("try_cast(orcamento_raw AS DECIMAL(18,2))"))
    .withColumn("receita_usd", F.expr("try_cast(receita_raw AS DECIMAL(18,2))"))
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
)
# Calcular os valores equivalentes em Reais (BRL) aplicando a taxa de cotação obtida
df_tratado = (df_tratado
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * taxa_dolar, 2))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * taxa_dolar, 2))
)
# Derivar Lucro (Dólar/Real) e Margem de Lucro % evitando divisão por zero
df_final = (df_tratado
    .withColumn("lucro_usd", F.round(F.col("receita_usd") - F.col("orcamento_usd"), 2))
    .withColumn("lucro_brl", F.round(F.col("receita_brl") - F.col("orcamento_brl"), 2))
    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("orcamento_usd") > 0,
            F.round((F.col("lucro_usd") / F.col("orcamento_usd")) * 100, 2)
        )
    )
    .select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_pct"
    )
)

# Gravação na Camada Silver
df_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")

# Exibe o resultado conferido
display(spark.table("silver.tb_financeiro_filmes").limit(20))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
293660,58000000.00,null,2.895882E8,null,null,null,null
299536,300000000.00,2052415039.00,1.49787E9,1.024750304822E10,1752415039.00,8.74963304822E9,584.14
299534,356000000.00,2800000000.00,1.7774724E9,1.398012E10,2444000000.00,1.22026476E10,686.52
475557,55000000.00,1074458282.00,2.746095E8,5.3646627562E9,1019458282.00,5.0900532562E9,1853.56
271110,250000000.00,null,1.248225E9,null,null,null,null
284054,200000000.00,1349926083.00,9.9858E8,6.74004593981E9,1149926083.00,5.74146593981E9,574.96
284052,180000000.00,676343174.00,8.98722E8,3.37691383346E9,496343174.00,2.47819183346E9,275.75
315635,175000000.00,880166924.00,8.737575E8,4.39458543484E9,705166924.00,3.52082793484E9,402.95
283995,200000000.00,863756051.00,9.9858E8,4.31264758704E9,663756051.00,3.31406758704E9,331.88
297761,175000000.00,746846894.00,8.737575E8,3.72893185705E9,571846894.00,2.85517435705E9,326.77


## **Camada Silver -  silver.tb_metricas_engajamento (origem: tb_movies_metrics)**

In [0]:
# Lê os dados de métricas da camada Bronze
df_met = spark.table("bronze.tb_movies_metrics")

# Exibe algumas linhas para verificarmos o formato dos valores monetários
display(df_met.limit(30))


df_met.printSchema()

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T13:11:33.153Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-16T13:11:33.153Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T13:11:33.153Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-16T13:11:33.153Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T13:11:33.153Z
284054,43.665,7.39,null,7.3,924922,2026-10-16T13:11:33.153Z
284052,70.535,7.427,20935,7.5,895880,2026-10-16T13:11:33.153Z
315635,65.88,7.345,20507,7.4,835116,2026-10-16T13:11:33.153Z
283995,67.553,7.624,20353,7.6,844767,2026-10-16T13:11:33.153Z
297761,35.356,5.909,20097,5.9,null,2026-10-16T13:11:33.153Z


root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_metricas_engajamento")

# Padronização de Colunas, Tratamento de Separadores e Conversão Segura de Tipos
df_tratado = df_met.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.expr("try_cast(replace(trim(popularity), ',', '.') AS DOUBLE)").alias("popularidade"),
    F.expr("try_cast(vote_average AS DOUBLE)").alias("nota_media_tmdb"),
    F.expr("try_cast(vote_count AS INT)").alias("qtd_votos_tmdb"),
    F.expr("try_cast(averageRating AS DOUBLE)").alias("nota_media_imdb"),
    F.expr("try_cast(numVotes AS INT)").alias("qtd_votos_imdb")
)

# Aplicação das Regras e Limites Numéricos de Negócio
# Notas médias (TMDB e IMDb) fora do intervalo [0, 10] são tratadas como NULL
# Contagens de votos ou índices de popularidade negativos são tratados como NULL

df_silver_metricas = (df_tratado
    # Validação de escala de notas (0 a 10)
    .withColumn(
        "nota_media_tmdb",
        F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb"))
    )
    # Validação de não negatividade
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb"))
    )
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") >= 0, F.col("popularidade"))
    )
)

# Gravação na Camada Silver 

df_silver_metricas.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")

# Exibição de amostra para validação
display(spark.table("silver.tb_metricas_engajamento").limit(20))



id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


## **Camada Silver -  silver.tb_avaliacoes_usuarios (origem: tb_movies_reviews)**

In [0]:
# Remove a tabela anterior, caso ela já exista
spark.sql("DROP TABLE IF EXISTS silver.tb_avaliacoes_usuarios")

# Carrega a tabela de avaliações da camada Bronze
df_avaliacoes = spark.table("bronze.tb_movies_reviews")

#  Visualiza alguns registros para identificar o formato dos dados
display(df_avaliacoes.limit(30))


df_avaliacoes.printSchema()

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T13:11:41.482Z
637007,Lucas Reis 602,3.9,null,2026-10-16T13:11:41.482Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T13:11:41.482Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T13:11:41.482Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T13:11:41.482Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-16T13:11:41.482Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-16T13:11:41.482Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-16T13:11:41.482Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-16T13:11:41.482Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-16T13:11:41.482Z


root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:

# Remove duplicatas integrais da base de origem
df_dedup = df_avaliacoes.dropDuplicates(["id", "nome", "nota", "comentario"])

# Mapeamento para nomes em português e tipagem
df_avaliacoes_final = df_dedup.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),

    # Validação de nota entre 0 e 10 (fora da faixa vira NULL)
    F.when(
        F.expr("try_cast(nota AS DOUBLE)").between(0, 10),
        F.expr("try_cast(nota AS DOUBLE)")
    ).alias("nota_usuario"),

    # Substitui comentários nulos ou só com espaços por "Sem comentário"
    F.when(
        F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""),
        "Sem comentário"
    ).otherwise(F.col("comentario")).alias("comentario_usuario")
)

# Gravação na Camada Silver
df_avaliacoes_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")

# Exibe conferência
display(spark.table("silver.tb_avaliacoes_usuarios").limit(20))


id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.
750569,Pedro Souza 611,4.4,Sem comentário
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.
823195,Rodrigo Mendes,2.9,Péssimo em todos os sentidos.
1055332,Gustavo Dias 303,null,Sem comentário
679864,Lucas Freitas 371,9.2,Perfeito! Um dos melhores filmes que já vi.


## **Camada Silver -  silver.tb_generos (origem: tb_credits_and_tags, coluna genres)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_generos")

# Carrega a tabela de generos e tags da camada Bronze
df_creditos = spark.table("bronze.tb_credits_and_tags")

# Visualiza alguns registros para identificar os formatos presentes na coluna genres
display(df_creditos.select("id", "genres").limit(30))

# Verifica os tipos das colunas da tabela
df_creditos.printSchema()

id,genres
293660,"Action, Adventure, Comedy"
299536,"Adventure, Action, Science Fiction"
299534,"Adventure, Science Fiction, Action"
475557,"Crime, Thriller, Drama"
271110,"Adventure, Action, Science Fiction"
284054,"Action, Adventure, Science Fiction"
284052,"Action, Adventure, Fantasy"
315635,"Action, Adventure, Science Fiction, Drama"
283995,"Science Fiction, Adventure, Action"
297761,Action|Adventure|Fantasy


root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Normaliza os separadores da coluna genres
df_genres = df_creditos.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.regexp_replace(F.col("genres"), r"[;|]", ",").alias("genres_normalized")
)

# Divide os gêneros e cria um registro para cada gênero
df_genres = (df_genres
    .withColumn("genero_array", F.split(F.col("genres_normalized"), ","))
    .withColumn("nome_genero", F.explode(F.col("genero_array")))
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
)

# Padroniza a capitalização antes de comparar com a lista
df_genres = df_genres.withColumn("nome_genero", F.initcap(F.col("nome_genero")))

# Domínio oficial de gêneros (19 gêneros do TMDB)
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "Tv Movie", "Thriller", "War", "Western"
]

# Filtra estritamente o domínio de negócio
df_genres = df_genres.filter(F.col("nome_genero").isin(generos_validos))

# Seleciona apenas as colunas oficiais da Silver
df_generos_final = df_genres.select("id_filme","nome_genero")

# Remove duplicatas
df_generos_final = df_generos_final.dropDuplicates(["id_filme", "nome_genero"])

# Salva na camada Silver
df_generos_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")

display(spark.table("silver.tb_generos").limit(20))


id_filme,nome_genero
335984,Science Fiction
281338,Science Fiction
512200,Adventure
38700,Crime
646380,Science Fiction
404368,Comedy
376867,Drama
616037,Fantasy
68735,Action
320288,Action


## **Camada Silver -  silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers,production_companies)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_pessoas_empresas")

# Carrega a tabela de pessoas e tags da camada Bronze
df_credits = spark.table("bronze.tb_credits_and_tags")



In [0]:
# Mapeia coluna de origem para tipo de entidade
mapping = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora"
}

dfs = []

for coluna, tipo in mapping.items():

    # Limpa colchetes, aspas e padroniza separadores (como ';' para ',')
    col_limpa = F.regexp_replace(F.col(coluna), r"[\[\]'\"]", "")
    col_limpa = F.regexp_replace(col_limpa, r"[;|]", ",")

    df = (df_creditos
        .select(
            F.expr("try_cast(id AS INT)").alias("id_filme"),
            col_limpa.alias("nome_entidade_raw")
        )
        # Desmembra os múltiplos elementos
        .withColumn("nome_entidade", F.explode(F.split("nome_entidade_raw", ",")))
        .withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade"))))
        .withColumn("tipo_entidade", F.lit(tipo))
        # Filtra resíduos, nulos e ruídos numéricos
        .filter(
            F.col("nome_entidade").isNotNull() & 
            (F.col("nome_entidade") != "") &
            (~F.lower(F.col("nome_entidade")).isin("n/a", "na", "none", "unknown", "null")) &
            (~F.col("nome_entidade").rlike(r"^[0-9.]+$"))
        )
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )

    dfs.append(df)

# Junta os 4 tipos de entidade
df_pessoas_empresas = dfs[0]
for df in dfs[1:]:
    df_pessoas_empresas = df_pessoas_empresas.unionByName(df)

# Remove duplicatas
df_pessoas_empresas_final = df_pessoas_empresas.dropDuplicates(
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

# Salva na Camada Silver
df_pessoas_empresas_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")

# Exibe conferência
display(spark.table("silver.tb_pessoas_empresas").limit(20))


id_filme,nome_entidade,tipo_entidade
634649,Jamie Foxx,Ator
383498,Leslie Uggams,Ator
497698,Ray Winstone,Ator
399566,Millie Bobby Brown,Ator
466282,Lana Condor,Ator
524434,Lia Mchugh,Ator
406997,Bryce Gheisar,Ator
398978,Joe Pesci,Ator
497582,Susan Wokoma,Ator
505642,Julia Louis-dreyfus,Ator


## **Camada Silver -  silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS silver.tb_cotacao_dolar")

# Lê a tabela Bronze
df_cotacao = spark.table("bronze.tb_cotacao_dolar")

# display(spark.table("bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

In [0]:
# Converte a dataHoraCotacao (string/timestamp) para apenas data (DATE)
df_cotacao = df_cotacao.withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))

# Cria a série temporal contínua (todos os dias entre a menor e a maior data)
datas = df_cotacao.select(
    F.min("data_cotacao").alias("data_inicial"),
    F.max("data_cotacao").alias("data_final")
).select(
    F.explode(
        F.sequence(
            F.col("data_inicial"),
            F.col("data_final"),
            F.expr("interval 1 day")
        )
    ).alias("data_cotacao")
)

# Junta o calendário completo com as cotações reais da Bronze (Left Join)
df_cotacao_serie = datas.join(
    df_cotacao.select("data_cotacao", "cotacaoCompra").dropDuplicates(["data_cotacao"]),
    on="data_cotacao",
    how="left"
)

# Aplica a técnica de Forward Fill com Window
# Pega o último valor válido anterior (ignorenulls=True) para preencher finais de semana e feriados
janela_ffill = (Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_final = (df_cotacao_serie
    .withColumn(
        "cotacao_compra",
        F.last("cotacaoCompra", ignorenulls=True).over(janela_ffill)
    )
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

# Salva a tabela na camada Silver
df_cotacao_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")

# Exibe a tabela ordenada para conferir os dias úteis e finais de semana preenchidos
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692
2026-10-07,4.9929
